# Proprioceptive sensing — object stiffness estimation

The hand squeezes a grasped object with the four fingers held at constant stiffness
$k_{\rm hold}$ = {K_TIP_HOLD} N/m (clamping side).  The thumb alone sweeps through
increasing tip stiffness levels and the resulting position–force finite difference
estimates object compliance without external force sensors.

---

## Why $\Delta x / \Delta F$ recovers $C_O$

At equilibrium, the VMC spring force equals the object reaction force.  Taking the
finite difference between the gentle baseline ($k_1$) and each probe level ($k_2 > k_1$):

$$\Delta x_\text{thumb} = x(k_2) - x(k_1), \qquad \Delta F_\text{thumb} = F(k_2) - F(k_1)$$

$$C_O = \frac{\|\Delta x_\text{thumb}\|}{\|\Delta F_\text{thumb}\|}$$

The hand compliance cancels exactly; lower $C_O$ means stiffer object.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import os, sys

sys.path.insert(0, os.path.join('../..'))
sys.path.insert(0, '.')
from hand_config import K_TIP_GENTLE, K_TIP_SWEEP, K_TIP_HOLD, N_RUNS, OBJECTS, FINGERTIPS

OUTPUT_DIR = os.path.join('outputs', 'object_stiffness_hand')

K_TIP_GENTLE = float(K_TIP_GENTLE)
K_TIP_SWEEP  = [float(k) for k in K_TIP_SWEEP]

OBJECT_LABELS = ['Sphere', 'Sponge', 'Yarn']
OBJ_COLORS    = {'hard': '#D55E00', 'medium': '#E69F00', 'soft': '#56B4E9'}

In [ ]:
def load_runs(obj):
    """Load all run CSV files for an object; fall back to the single-file (video) naming."""
    dfs = []
    for r in range(1, N_RUNS + 1):
        p = os.path.join(OUTPUT_DIR, f'object_stiffness_hand_{obj}_run{r}.csv')
        if os.path.exists(p):
            dfs.append(pd.read_csv(p))
    if not dfs:
        p = os.path.join(OUTPUT_DIR, f'object_stiffness_hand_{obj}.csv')
        if os.path.exists(p):
            dfs.append(pd.read_csv(p))
    return dfs


def _thumb_baseline(df):
    gentle = df[df['k_tip_Npm'] == K_TIP_GENTLE]
    pos_b  = gentle[['tip_thumb_x_m', 'tip_thumb_y_m', 'tip_thumb_z_m']].median().to_numpy()
    F_b    = gentle[['force_1st_thumb_x_N', 'force_1st_thumb_y_N', 'force_1st_thumb_z_N']].median().to_numpy()
    return pos_b, F_b


def compliance_per_run(dfs):
    """Median-based C_O [m/N] for each run at each sweep level.
    Returns dict: k -> 1-D array of length n_runs."""
    out = {k: [] for k in K_TIP_SWEEP}
    for df in dfs:
        pos_b, F_b = _thumb_baseline(df)
        for k in K_TIP_SWEEP:
            sub  = df[df['k_tip_Npm'] == k]
            pos  = sub[['tip_thumb_x_m', 'tip_thumb_y_m', 'tip_thumb_z_m']].median().to_numpy()
            F1   = sub[['force_1st_thumb_x_N', 'force_1st_thumb_y_N', 'force_1st_thumb_z_N']].median().to_numpy()
            nF   = np.linalg.norm(F1 - F_b)
            out[k].append(np.linalg.norm(pos - pos_b) / nF if nF > 1e-12 else np.nan)
    return {k: np.array(v) for k, v in out.items()}


def displacement_per_run(dfs):
    """Median thumb displacement [m] from baseline per run at each sweep level."""
    out = {k: [] for k in K_TIP_SWEEP}
    for df in dfs:
        pos_b, _ = _thumb_baseline(df)
        for k in K_TIP_SWEEP:
            sub = df[df['k_tip_Npm'] == k]
            pos = sub[['tip_thumb_x_m', 'tip_thumb_y_m', 'tip_thumb_z_m']].median().to_numpy()
            out[k].append(np.linalg.norm(pos - pos_b))
    return {k: np.array(v) for k, v in out.items()}


def force_per_run(dfs):
    """Median thumb force change [N] from baseline per run at each sweep level."""
    out = {k: [] for k in K_TIP_SWEEP}
    for df in dfs:
        _, F_b = _thumb_baseline(df)
        for k in K_TIP_SWEEP:
            sub = df[df['k_tip_Npm'] == k]
            F1  = sub[['force_1st_thumb_x_N', 'force_1st_thumb_y_N', 'force_1st_thumb_z_N']].median().to_numpy()
            out[k].append(np.linalg.norm(F1 - F_b))
    return {k: np.array(v) for k, v in out.items()}


# ── Load data ──────────────────────────────────────────────────────────────────
data = {}
print('Loading data:')
for obj in OBJECTS:
    dfs = load_runs(obj)
    if not dfs:
        print(f'  {obj:<10s}: no data')
        continue
    data[obj] = dict(
        dfs        = dfs,
        compliance = compliance_per_run(dfs),
        displ      = displacement_per_run(dfs),
        force      = force_per_run(dfs),
    )
    n = len(dfs)
    c_top     = data[obj]['compliance'][max(K_TIP_SWEEP)]
    c_avg_all = np.concatenate([data[obj]['compliance'][k] for k in K_TIP_SWEEP])
    runs_str  = ', '.join(f'{v:.1f}' for v in c_top * 1e3)
    print(f'  {obj:<10s}: {n} run(s)  C_O at K_top: {np.nanmean(c_top)*1e3:.1f} ± {np.nanstd(c_top)*1e3:.2f} mm/N  [{runs_str}]  |  C_O avg all k: {np.nanmean(c_avg_all)*1e3:.1f} ± {np.nanstd(c_avg_all)*1e3:.2f} mm/N')

## Thumb object compliance $C_O$ vs $k_{tip}$

$C_O = \|\Delta x_{\rm thumb}\| / \|\Delta F_{\rm thumb}\|$ — lower = stiffer object. Dashed lines = individual trials, solid = mean, band = ±1 std.

In [ ]:
def _sweep_plot(ax, per_run_dict, scale, obj, ylabel):
    """Plot per-run dashed lines + mean solid + std band for one object."""
    k_levels = sorted(per_run_dict.keys())
    vals     = np.array([per_run_dict[k] * scale for k in k_levels])  # (n_k, n_runs)
    mean     = np.nanmean(vals, axis=1)
    std      = np.nanstd(vals,  axis=1)
    color    = OBJ_COLORS[obj]
    name     = OBJECT_LABELS[OBJECTS.index(obj)] if obj in OBJECTS else obj
    # individual runs
    for r in range(vals.shape[1]):
        ax.plot(k_levels, vals[:, r], color=color, lw=1.0,
                linestyle='--', alpha=0.25)
    # mean ± std
    ax.fill_between(k_levels, mean - std, mean + std, color=color, alpha=0.15)
    ax.plot(k_levels, mean, color=color, lw=2.5, linestyle='-', label=name)

fig, ax = plt.subplots(figsize=(7, 4))
for obj in OBJECTS:
    if obj not in data:
        continue
    _sweep_plot(ax, data[obj]['compliance'], 1e3, obj, r'$C_O^{\mathrm{thumb}}$ [mm/N]')
ax.set_xlabel(r'$k_{tip}$ [N/m]')
ax.set_ylabel(r'$C_O^{\mathrm{thumb}}$ [mm/N]')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)
fig.tight_layout()
os.makedirs(OUTPUT_DIR, exist_ok=True)
fig.savefig(os.path.join(OUTPUT_DIR, 'thumb_compliance_sweep.pdf'), bbox_inches='tight')
plt.show()

## Thumb tip displacement $\|\Delta x_{\rm thumb}\|$ vs $k_{tip}$

In [ ]:
def _sweep_plot(ax, per_run_dict, scale, obj, ylabel):
    """Plot per-run dashed lines + mean solid + std band for one object."""
    k_levels = sorted(per_run_dict.keys())
    vals     = np.array([per_run_dict[k] * scale for k in k_levels])  # (n_k, n_runs)
    mean     = np.nanmean(vals, axis=1)
    std      = np.nanstd(vals,  axis=1)
    color    = OBJ_COLORS[obj]
    name     = OBJECT_LABELS[OBJECTS.index(obj)] if obj in OBJECTS else obj
    # individual runs
    for r in range(vals.shape[1]):
        ax.plot(k_levels, vals[:, r], color=color, lw=1.0,
                linestyle='--', alpha=0.25)
    # mean ± std
    ax.fill_between(k_levels, mean - std, mean + std, color=color, alpha=0.15)
    ax.plot(k_levels, mean, color=color, lw=2.5, linestyle='-', label=name)

fig, ax = plt.subplots(figsize=(7, 4))
for obj in OBJECTS:
    if obj not in data:
        continue
    _sweep_plot(ax, data[obj]['displ'], 1e3, obj, r'$\|\Delta x\|$ [mm]')
ax.set_xlabel(r'$k_{tip}$ [N/m]')
ax.set_ylabel(r'$\|\Delta x_{\mathrm{thumb}}\|$ [mm]')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'thumb_displacement_sweep.pdf'), bbox_inches='tight')
plt.show()

## Thumb contact force $\|\Delta F_{\rm thumb}\|$ vs $k_{tip}$

In [ ]:
def _sweep_plot(ax, per_run_dict, scale, obj, ylabel):
    """Plot per-run dashed lines + mean solid + std band for one object."""
    k_levels = sorted(per_run_dict.keys())
    vals     = np.array([per_run_dict[k] * scale for k in k_levels])  # (n_k, n_runs)
    mean     = np.nanmean(vals, axis=1)
    std      = np.nanstd(vals,  axis=1)
    color    = OBJ_COLORS[obj]
    name     = OBJECT_LABELS[OBJECTS.index(obj)] if obj in OBJECTS else obj
    # individual runs
    for r in range(vals.shape[1]):
        ax.plot(k_levels, vals[:, r], color=color, lw=1.0,
                linestyle='--', alpha=0.25)
    # mean ± std
    ax.fill_between(k_levels, mean - std, mean + std, color=color, alpha=0.15)
    ax.plot(k_levels, mean, color=color, lw=2.5, linestyle='-', label=name)

fig, ax = plt.subplots(figsize=(7, 4))
for obj in OBJECTS:
    if obj not in data:
        continue
    _sweep_plot(ax, data[obj]['force'], 1.0, obj, r'$\|\Delta F\|$ [N]')
ax.set_xlabel(r'$k_{tip}$ [N/m]')
ax.set_ylabel(r'$\|\Delta F_{\mathrm{thumb}}\|$ [N]')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.4)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'thumb_force_sweep.pdf'), bbox_inches='tight')
plt.show()

## Object classification by thumb compliance

$C_O^{\rm thumb}$ at the highest $k_{tip}$ — each trial shown as a dot, filled marker = mean ± std. Lower = stiffer.

In [ ]:
K_TOP = max(K_TIP_SWEEP)

C_O_runs = {obj: data[obj]['compliance'][K_TOP] * 1e3
            for obj in OBJECTS if obj in data and K_TOP in data[obj]['compliance']}

if not C_O_runs:
    print('No data — skipping classification.')
else:
    sorted_objs = sorted(C_O_runs, key=lambda o: np.nanmean(C_O_runs[o]))
    cluster_labels = {sorted_objs[i]: lbl
                      for i, lbl in enumerate(['stiff', 'medium', 'soft'][:len(sorted_objs)])}
    means = [np.nanmean(C_O_runs[o]) for o in sorted_objs]
    thresholds = [(means[i] + means[i+1]) / 2 for i in range(len(means) - 1)]

    fig, ax = plt.subplots(figsize=(7, 3))
    for obj in sorted_objs:
        vals  = C_O_runs[obj]
        m, s  = float(np.nanmean(vals)), float(np.nanstd(vals))
        color = OBJ_COLORS[obj]
        oi    = OBJECTS.index(obj)
        name  = OBJECT_LABELS[oi] if oi < len(OBJECT_LABELS) else obj
        lbl   = cluster_labels[obj]
        # individual trial dots
        ax.scatter(vals, np.zeros_like(vals), color=color, s=40, alpha=0.4, zorder=4)
        # mean with std error bar
        ax.errorbar([m], [0], xerr=[[s], [s]],
                    fmt='o', color=color, ms=12, capsize=6, lw=2, zorder=5)
        ax.annotate(f'{name}\n({lbl})', (m, 0),
                    textcoords='offset points', xytext=(0, 24),
                    ha='center', fontsize=11)
    for thr in thresholds:
        ax.axvline(thr, color='0.4', lw=1.2, linestyle='--')

    ax.set_xlabel(r'$C_O^{\mathrm{thumb}}$ at $k_{tip}=' + f'{K_TOP:.0f}$'
                  + r' N/m  [mm/N]  (mean $\pm$ std, dots = trials)')
    ax.set_yticks([])
    ax.spines[['left', 'top', 'right']].set_visible(False)
    fig.tight_layout()
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    fig.savefig(os.path.join(OUTPUT_DIR, 'object_classification.pdf'), bbox_inches='tight')
    plt.show()

    print('Object classification:')
    for obj in sorted_objs:
        oi    = OBJECTS.index(obj)
        name  = OBJECT_LABELS[oi] if oi < len(OBJECT_LABELS) else obj
        m, s  = np.nanmean(C_O_runs[obj]), np.nanstd(C_O_runs[obj])
        print(f'  {name:<10s}  {m:.2f} ± {s:.3f} mm/N  --> {cluster_labels[obj]}')

---

## Dynamic stiffness ramp — transient response

The thumb tip stiffness is ramped from `K_TIP_GENTLE` → `K_TIP_PROBE` at different
speeds (`RAMP_DURATIONS`).  The four clamping fingers stay frozen at `K_TIP_HOLD`.
Each CSV contains three phases: `baseline` (before ramp), `ramp` (during the K step),
`post_ramp` (after reaching `K_TIP_PROBE`).

Time is zeroed at the start of the baseline window for each condition.


In [ ]:
from glob import glob
from hand_config import K_TIP_PROBE, RAMP_DURATIONS, POST_RAMP_DURATION, BASELINE_DURATION

DYN_DIR = os.path.join('outputs', 'object_stiffness_dynamic')

# Ramp-duration label for plots (ms)
DUR_LABELS  = [f'{int(round(d * 1000))} ms' for d in RAMP_DURATIONS]
DUR_COLORS  = plt.cm.viridis(np.linspace(0.15, 0.90, len(RAMP_DURATIONS)))


def load_dynamic(obj):
    """Return dict dur_s -> list[DataFrame], one df per run."""
    out = {}
    for dur in RAMP_DURATIONS:
        tag = f'{int(round(dur * 1000))}ms'
        dfs = []
        # multi-run files first
        for r in range(1, N_RUNS + 1):
            p = os.path.join(DYN_DIR, f'object_stiffness_dynamic_{obj}_dur{tag}_run{r}.csv')
            if os.path.exists(p):
                dfs.append(pd.read_csv(p))
        # single-run fallback
        if not dfs:
            p = os.path.join(DYN_DIR, f'object_stiffness_dynamic_{obj}_dur{tag}.csv')
            if os.path.exists(p):
                dfs.append(pd.read_csv(p))
        if dfs:
            out[dur] = dfs
    return out


dyn_data = {}
print('Loading dynamic data:')
for obj in OBJECTS:
    d = load_dynamic(obj)
    if d:
        dyn_data[obj] = d
        counts = {dur: len(dfs) for dur, dfs in d.items()}
        print(f'  {obj:<10s}: {counts}')
    else:
        print(f'  {obj:<10s}: no data')


### Thumb displacement time-series during ramp

Each panel = one object.  Lines = ramp durations.  The ramp window is shaded;
time 0 = start of the baseline.


In [ ]:
def _dyn_timeseries(ax, dfs_by_dur, col, scale=1.0, ylabel=''):
    """Overlay mean trajectory for each ramp duration.

    col can be a single column name or a list of column names; in the latter case
    the Euclidean norm across columns is used (e.g. for 3-D displacement magnitude).
    """
    for i, dur in enumerate(RAMP_DURATIONS):
        if dur not in dfs_by_dur:
            continue
        t_end = BASELINE_DURATION + dur + POST_RAMP_DURATION
        t_grid = np.linspace(0, t_end, 500)
        traces = []
        for df in dfs_by_dur[dur]:
            t = df['time_s'].to_numpy()
            if isinstance(col, (list, tuple)):
                val = np.linalg.norm(df[list(col)].to_numpy(), axis=1) * scale
            else:
                val = df[col].to_numpy() * scale
            traces.append(np.interp(t_grid, t, val))
        mean = np.nanmean(traces, axis=0)
        std  = np.nanstd(traces,  axis=0)
        ax.fill_between(t_grid, mean - std, mean + std,
                        color=DUR_COLORS[i], alpha=0.18)
        ax.plot(t_grid, mean, color=DUR_COLORS[i], lw=2,
                label=DUR_LABELS[i])

    # shade ramp windows per duration
    for i, dur in enumerate(RAMP_DURATIONS):
        ax.axvspan(BASELINE_DURATION, BASELINE_DURATION + dur,
                   color=DUR_COLORS[i], alpha=0.06)

    ax.axvline(BASELINE_DURATION, color='0.5', lw=1, linestyle='--')
    ax.set_xlabel('time [s]')
    ax.set_ylabel(ylabel)
    ax.legend(title='ramp dur.', fontsize=9, frameon=False)
    ax.spines[['top', 'right']].set_visible(False)


_DISP_COLS  = ['disp_thumb_x_m', 'disp_thumb_y_m', 'disp_thumb_z_m']

if dyn_data:
    n_obj = len([o for o in OBJECTS if o in dyn_data])
    fig, axes = plt.subplots(1, n_obj, figsize=(5 * n_obj, 4), sharey=True)
    if n_obj == 1:
        axes = [axes]
    for ax, obj in zip(axes, [o for o in OBJECTS if o in dyn_data]):
        _dyn_timeseries(ax, dyn_data[obj], _DISP_COLS, scale=1e3,
                        ylabel=r'$\|\Delta x_\mathrm{thumb}\|$ [mm]')
        ax.set_title(OBJECT_LABELS[OBJECTS.index(obj)])
    fig.suptitle('Thumb displacement — dynamic ramp', y=1.02)
    fig.tight_layout()
    os.makedirs(DYN_DIR, exist_ok=True)
    fig.savefig(os.path.join(DYN_DIR, 'dyn_displacement_timeseries.pdf'), bbox_inches='tight')
    plt.show()
else:
    print('No dynamic data found.')


### Thumb contact force time-series during ramp


In [ ]:
if dyn_data:
    n_obj = len([o for o in OBJECTS if o in dyn_data])
    fig, axes = plt.subplots(1, n_obj, figsize=(5 * n_obj, 4), sharey=True)
    if n_obj == 1:
        axes = [axes]
    for ax, obj in zip(axes, [o for o in OBJECTS if o in dyn_data]):
        _dyn_timeseries(ax, dyn_data[obj],
                        'force_1st_thumb_mag_N', scale=1.0,
                        ylabel=r'$\|F_\mathrm{thumb}\|$ [N]')
        ax.set_title(OBJECT_LABELS[OBJECTS.index(obj)])
    fig.suptitle('Thumb contact force — dynamic ramp', y=1.02)
    fig.tight_layout()
    fig.savefig(os.path.join(DYN_DIR, 'dyn_force_timeseries.pdf'), bbox_inches='tight')
    plt.show()


### Settling time vs ramp duration

95 % settling time: first time after the ramp ends where the thumb displacement
stays within 5 % of its post-ramp mean for at least 0.2 s.


In [ ]:
def _settling_time(df, tol=0.05, hold=0.2):
    """Time from ramp-end until displacement (relative to baseline) stays
    within tol of its post-ramp median for at least `hold` seconds.
    Returns NaN if convergence is not reached within POST_RAMP_DURATION.
    """
    base = df[df['phase'] == 'baseline']
    post = df[df['phase'] == 'post_ramp']
    if base.empty or post.empty:
        return np.nan
    pos_b    = base[['tip_thumb_x_m', 'tip_thumb_y_m', 'tip_thumb_z_m']].median().to_numpy()
    pos_post = post[['tip_thumb_x_m', 'tip_thumb_y_m', 'tip_thumb_z_m']].to_numpy()
    disp_mag = np.linalg.norm(pos_post - pos_b, axis=1)
    ss_val   = np.median(disp_mag)
    if ss_val < 1e-9:
        return np.nan
    band    = tol * ss_val
    t0_post = post['time_s'].iloc[0]
    t       = post['time_s'].to_numpy()
    for j in range(len(t)):
        window = disp_mag[(t >= t[j]) & (t <= t[j] + hold)]
        if len(window) > 0 and np.all(np.abs(window - ss_val) < band):
            return t[j] - t0_post
    return np.nan


if dyn_data:
    fig, ax = plt.subplots(figsize=(7, 4))
    for obj in OBJECTS:
        if obj not in dyn_data:
            continue
        color = OBJ_COLORS[obj]
        name  = OBJECT_LABELS[OBJECTS.index(obj)]
        settle_mean, settle_std = [], []
        for dur in RAMP_DURATIONS:
            if dur not in dyn_data[obj]:
                settle_mean.append(np.nan)
                settle_std.append(np.nan)
                continue
            vals = [_settling_time(df) for df in dyn_data[obj][dur]]
            settle_mean.append(np.nanmean(vals))
            settle_std.append(np.nanstd(vals))
        ax.errorbar(RAMP_DURATIONS, settle_mean, yerr=settle_std,
                    fmt='o-', color=color, capsize=4, lw=2, label=name)
    ax.set_xlabel('ramp duration [s]')
    ax.set_ylabel('95 % settling time after ramp end [s]')
    ax.legend(frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    fig.tight_layout()
    fig.savefig(os.path.join(DYN_DIR, 'dyn_settling_time.pdf'), bbox_inches='tight')
    plt.show()


### Steady-state compliance $C_O$ from dynamic ramp

Same finite-difference estimator as the static experiment, applied to the
post-ramp steady state — lets you compare dynamic vs static $C_O$ estimates.


In [ ]:
def _dyn_compliance(dfs_by_dur):
    """C_O [m/N] using peak displacement and peak force during ramp+post_ramp."""
    out = {}
    for dur, dfs in dfs_by_dur.items():
        co_list = []
        for df in dfs:
            base   = df[df['phase'] == 'baseline']
            active = df[df['phase'].isin(['ramp', 'post_ramp'])]
            if base.empty or active.empty:
                co_list.append(np.nan)
                continue
            pos_b = base[['tip_thumb_x_m', 'tip_thumb_y_m', 'tip_thumb_z_m']].median().to_numpy()
            F_b   = base[['force_1st_thumb_x_N', 'force_1st_thumb_y_N', 'force_1st_thumb_z_N']].median().to_numpy()
            disp_mag  = np.linalg.norm(
                active[['tip_thumb_x_m', 'tip_thumb_y_m', 'tip_thumb_z_m']].to_numpy() - pos_b,
                axis=1)
            force_mag = np.linalg.norm(
                active[['force_1st_thumb_x_N', 'force_1st_thumb_y_N', 'force_1st_thumb_z_N']].to_numpy() - F_b,
                axis=1)
            peak_disp  = disp_mag.max()
            peak_force = force_mag.max()
            co_list.append(peak_disp / peak_force if peak_force > 1e-12 else np.nan)
        out[dur] = np.array(co_list)
    return out


if dyn_data:
    fig, ax = plt.subplots(figsize=(7, 4))
    for obj in OBJECTS:
        if obj not in dyn_data:
            continue
        color = OBJ_COLORS[obj]
        name  = OBJECT_LABELS[OBJECTS.index(obj)]
        co    = _dyn_compliance(dyn_data[obj])
        means = [np.nanmean(co[d]) * 1e3 if d in co else np.nan for d in RAMP_DURATIONS]
        stds  = [np.nanstd(co[d])  * 1e3 if d in co else np.nan for d in RAMP_DURATIONS]
        ax.errorbar(RAMP_DURATIONS, means, yerr=stds,
                    fmt='o-', color=color, capsize=4, lw=2, label=name)
    ax.set_xlabel('ramp duration [s]')
    ax.set_ylabel(r'$C_O^{\mathrm{dyn}}$ [mm/N]')
    ax.legend(frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    fig.tight_layout()
    fig.savefig(os.path.join(DYN_DIR, 'dyn_compliance_vs_ramp_dur.pdf'), bbox_inches='tight')
    plt.show()
